## 1. Load Dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = "../data/raw/DataCoSupplyChainDataset.csv"

df = pd.read_csv(
    file_path,
    encoding="latin1"
)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 180,519
Columns: 53


In [3]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

## 2. Remove Irrelevant & Sensitive Columns

In [5]:
columns_to_drop = [
    "Product Description",
    "Order Zipcode",
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Product Image"
]

In [6]:
df = df.drop(columns=columns_to_drop)

print(f"Shape after removing irrelevant columns: {df.shape}")

Shape after removing irrelevant columns: (180519, 45)


In [7]:
constant_columns = [
    column
    for column in df.columns
    if df[column].nunique(dropna=False) <= 1
]

print("Constant columns:")
print(constant_columns)

Constant columns:
['Product Status']


In [8]:
df[constant_columns].nunique(dropna=False)

Product Status    1
dtype: int64

In [9]:
df = df.drop(columns=constant_columns)

print(f"Shape after removing constant columns: {df.shape}")

Shape after removing constant columns: (180519, 44)


## 3. Handle Missing Values

In [10]:
missing_summary_cleaning = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().sum() / len(df) * 100
    )
})

missing_summary_cleaning = (
    missing_summary_cleaning[
        missing_summary_cleaning["missing_count"] > 0
    ]
    .sort_values("missing_count", ascending=False)
)

missing_summary_cleaning

,missing_count,missing_percentage
Customer Zipcode,3,0.001662


In [11]:
print(f"Columns with missing values: {len(missing_summary_cleaning)}")
print(f"Total missing values: {df.isnull().sum().sum():,}")

Columns with missing values: 1
Total missing values: 3


`customer_zipcode` contains 3 missing values (0.00166% of records). The rows are retained because the missing values are negligible and customer ZIP code is not a core analytical field. No artificial imputation is applied to avoid introducing potentially incorrect geographic information.

## 4. Convert Date Columns

In [12]:
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

print("Date column data types:")
print(df[date_columns].dtypes)

Date column data types:
order date (DateOrders)       datetime64[us]
shipping date (DateOrders)    datetime64[us]
dtype: object


In [13]:
print("\nInvalid datetime values:")
print(df[date_columns].isna().sum())


Invalid datetime values:
order date (DateOrders)       0
shipping date (DateOrders)    0
dtype: int64


In [14]:
print("\nDate range:")
for col in date_columns:
    print(
        f"{col}: "
        f"{df[col].min()} → {df[col].max()}"
    )


Date range:
order date (DateOrders): 2015-01-01 00:00:00 → 2018-01-31 23:38:00
shipping date (DateOrders): 2015-01-03 00:00:00 → 2018-02-06 22:14:00


In [15]:
invalid_shipping_before_order = (
    df["shipping date (DateOrders)"]
    < df["order date (DateOrders)"]
).sum()

print(
    "Shipping date before order date:",
    invalid_shipping_before_order
)

Shipping date before order date: 0


## 4. Data Type Optimization

In [16]:
print("Current data types:")
print(df.dtypes)

Current data types:
Type                                        str
Days for shipping (real)                  int64
Days for shipment (scheduled)             int64
Benefit per order                       float64
Sales per customer                      float64
Delivery Status                             str
Late_delivery_risk                        int64
Category Id                               int64
Category Name                               str
Customer City                               str
Customer Country                            str
Customer Id                               int64
Customer Segment                            str
Customer State                              str
Customer Zipcode                        float64
Department Id                             int64
Department Name                             str
Latitude                                float64
Longitude                               float64
Market                                      str
Order City          

In [17]:
categorical_columns = [
    "Type",
    "Delivery Status",
    "Category Name",
    "Customer Country",
    "Customer Segment",
    "Customer State",
    "Department Name",
    "Market",
    "Order Country",
    "Order Region",
    "Order Status",
    "Shipping Mode"
]

for col in categorical_columns:
    df[col] = df[col].astype("category")

In [18]:
df["Late_delivery_risk"] = df["Late_delivery_risk"].astype("int8")

In [21]:
print("Final data types:")
print(df.dtypes)

print("\nData type summary:")
print(df.dtypes.value_counts())

Final data types:
Type                                   category
Days for shipping (real)                  int64
Days for shipment (scheduled)             int64
Benefit per order                       float64
Sales per customer                      float64
Delivery Status                        category
Late_delivery_risk                         int8
Category Id                               int64
Category Name                          category
Customer City                               str
Customer Country                       category
Customer Id                               int64
Customer Segment                       category
Customer State                         category
Customer Zipcode                        float64
Department Id                             int64
Department Name                        category
Latitude                                float64
Longitude                               float64
Market                                 category
Order City            

In [22]:
validation_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Late_delivery_risk",
    "Type",
    "Delivery Status",
    "Category Name",
    "Customer Segment",
    "Market",
    "Order Status",
    "Shipping Mode"
]

print("\nValidation:")
print(df[validation_columns].dtypes)


Validation:
order date (DateOrders)       datetime64[us]
shipping date (DateOrders)    datetime64[us]
Late_delivery_risk                      int8
Type                                category
Delivery Status                     category
Category Name                       category
Customer Segment                    category
Market                              category
Order Status                        category
Shipping Mode                       category
dtype: object


## 5. Standardize Column Names

In [23]:
# Standardize column names to snake_case

column_mapping = {
    "Type": "payment_type",
    "Days for shipping (real)": "days_for_shipping_real",
    "Days for shipment (scheduled)": "days_for_shipment_scheduled",
    "Benefit per order": "benefit_per_order",
    "Sales per customer": "sales_per_customer",
    "Delivery Status": "delivery_status",
    "Late_delivery_risk": "late_delivery_risk",
    "Category Id": "category_id",
    "Category Name": "category_name",
    "Customer City": "customer_city",
    "Customer Country": "customer_country",
    "Customer Id": "customer_id",
    "Customer Segment": "customer_segment",
    "Customer State": "customer_state",
    "Customer Zipcode": "customer_zipcode",
    "Department Id": "department_id",
    "Department Name": "department_name",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Market": "market",
    "Order City": "order_city",
    "Order Country": "order_country",
    "Order Customer Id": "order_customer_id",
    "order date (DateOrders)": "order_date",
    "Order Id": "order_id",
    "Order Item Cardprod Id": "order_item_cardprod_id",
    "Order Item Discount": "order_item_discount",
    "Order Item Discount Rate": "order_item_discount_rate",
    "Order Item Id": "order_item_id",
    "Order Item Product Price": "order_item_product_price",
    "Order Item Profit Ratio": "order_item_profit_ratio",
    "Order Item Quantity": "order_item_quantity",
    "Sales": "sales",
    "Order Item Total": "order_item_total",
    "Order Profit Per Order": "order_profit_per_order",
    "Order Region": "order_region",
    "Order State": "order_state",
    "Order Status": "order_status",
    "Product Card Id": "product_card_id",
    "Product Category Id": "product_category_id",
    "Product Name": "product_name",
    "Product Price": "product_price",
    "shipping date (DateOrders)": "shipping_date",
    "Shipping Mode": "shipping_mode"
}

df = df.rename(columns=column_mapping)

In [24]:
print("Column names:")
print(df.columns.tolist())

Column names:
['payment_type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_category_id', 'product_name', 'product_price', 'shipping_date', 'shipping_mode']


## 6. Feature Engineering

### 6.1 Delivery Performance Features

In [25]:
df["shipping_delay_days"] = (
    df["days_for_shipping_real"]
    - df["days_for_shipment_scheduled"]
)

In [26]:
df["is_late_delivery"] = (
    df["shipping_delay_days"] > 0
).astype("int8")

In [27]:
df["shipping_performance"] = np.select(
    [
        df["shipping_delay_days"] > 0,
        df["shipping_delay_days"] == 0,
        df["shipping_delay_days"] < 0
    ],
    [
        "Late",
        "On Time",
        "Early"
    ],
    default="Unknown"
)

df["shipping_performance"] = df["shipping_performance"].astype("category")

In [28]:
df["shipping_delay_days"].describe()

count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: shipping_delay_days, dtype: float64

In [29]:
df["is_late_delivery"].value_counts()

is_late_delivery
1    103400
0     77119
Name: count, dtype: int64

In [30]:
df["shipping_performance"].value_counts()

shipping_performance
Late       103400
Early       43366
On Time     33753
Name: count, dtype: int64

### 6.2 Order Date & Time Features

In [31]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["order_quarter"] = df["order_date"].dt.quarter
df["order_day"] = df["order_date"].dt.day
df["order_day_of_week"] = df["order_date"].dt.dayofweek
df["order_week_of_year"] = df["order_date"].dt.isocalendar().week.astype("int8")

In [32]:
date_features = [
    "order_year",
    "order_month",
    "order_quarter",
    "order_day",
    "order_day_of_week",
    "order_week_of_year"
]

print("\nFeature ranges:")

for column in date_features:
    print(
        f"{column}: "
        f"{df[column].min()} → {df[column].max()}"
    )


Feature ranges:
order_year: 2015 → 2018
order_month: 1 → 12
order_quarter: 1 → 4
order_day: 1 → 31
order_day_of_week: 0 → 6
order_week_of_year: 1 → 53


## 7. Create Analytical Dataset

In [36]:
import os

processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

cleaned_file_path = os.path.join(
    processed_dir,
    "cleaned_supply_chain.csv"
)

df.to_csv(
    cleaned_file_path,
    index=False
)

print(f"Clean dataset saved to: {cleaned_file_path}")

Clean dataset saved to: ../data/processed\cleaned_supply_chain.csv


In [37]:
df_clean = pd.read_csv(
    cleaned_file_path,
    parse_dates=[
        "order_date",
        "shipping_date"
    ]
)

print(f"Rows: {df_clean.shape[0]:,}")
print(f"Columns: {df_clean.shape[1]}")

Rows: 180,519
Columns: 53


In [49]:
df_clean.columns.tolist()

['payment_type',
 'days_for_shipping_real',
 'days_for_shipment_scheduled',
 'benefit_per_order',
 'sales_per_customer',
 'delivery_status',
 'late_delivery_risk',
 'category_id',
 'category_name',
 'customer_city',
 'customer_country',
 'customer_id',
 'customer_segment',
 'customer_state',
 'customer_zipcode',
 'department_id',
 'department_name',
 'latitude',
 'longitude',
 'market',
 'order_city',
 'order_country',
 'order_customer_id',
 'order_date',
 'order_id',
 'order_item_cardprod_id',
 'order_item_discount',
 'order_item_discount_rate',
 'order_item_id',
 'order_item_product_price',
 'order_item_profit_ratio',
 'order_item_quantity',
 'sales',
 'order_item_total',
 'order_profit_per_order',
 'order_region',
 'order_state',
 'order_status',
 'product_card_id',
 'product_category_id',
 'product_name',
 'product_price',
 'shipping_date',
 'shipping_mode',
 'shipping_delay_days',
 'is_late_delivery',
 'shipping_performance',
 'order_year',
 'order_month',
 'order_quarter',
 'orde

In [40]:
missing_final = df_clean.isnull().sum()

missing_final = missing_final[
    missing_final > 0
].sort_values(ascending=False)

missing_final

customer_zipcode    3
dtype: int64

In [48]:
print("Exact duplicate rows:", df_clean.duplicated().sum())

Exact duplicate rows: 0
